In [12]:

import numpy as np
import pandas as pd

df = pd.read_csv("data/case1/train1.csv")
df["kilo_rpm"] = df["speed_rpm"] / 1000
dataset = df[["power_w"]].copy()

dataset['kilo_rpm'] = df["kilo_rpm"]
dataset['kilo_rpm * flow_m3_h'] = df["kilo_rpm"] * df["flow_m3_h"]
dataset['kilo_rpm * flow_m3_h**2'] = df["kilo_rpm"] * df["flow_m3_h"]**2
dataset['kilo_rpm * flow_m3_h**3'] = df["kilo_rpm"] * df["flow_m3_h"]**3

dataset['kilo_rpm**2'] = df["kilo_rpm"]**2
dataset['kilo_rpm**2 * flow_m3_h'] = df["kilo_rpm"]**2 * df["flow_m3_h"]
dataset['kilo_rpm**2 * flow_m3_h**2'] = df["kilo_rpm"]**2 * df["flow_m3_h"]**2
dataset['kilo_rpm**2 * flow_m3_h**3'] = df["kilo_rpm"]**2 * df["flow_m3_h"]**3

dataset['kilo_rpm**3'] = df["kilo_rpm"]**3
dataset['kilo_rpm**3 * flow_m3_h'] = df["kilo_rpm"]**3 * df["flow_m3_h"]
dataset['kilo_rpm**3 * flow_m3_h**2'] = df["kilo_rpm"]**3 * df["flow_m3_h"]**2
dataset['kilo_rpm**3 * flow_m3_h**3'] = df["kilo_rpm"]**3 * df["flow_m3_h"]**3

dataset["flow_m3_h"] = df["flow_m3_h"]
dataset["flow_m3_h**2"] = df["flow_m3_h"]**2
dataset["flow_m3_h**3"] = df["flow_m3_h"]**3

dataset["constant"] = 1

test_df = pd.read_csv("data/case1/test1.csv")
test_df["kilo_rpm"] = test_df["speed_rpm"] / 1000

test_dataset = test_df[["power_w"]].copy()

test_dataset['kilo_rpm'] = test_df["kilo_rpm"]
test_dataset['kilo_rpm * flow_m3_h'] = test_df["kilo_rpm"] * test_df["flow_m3_h"]
test_dataset['kilo_rpm * flow_m3_h**2'] = test_df["kilo_rpm"] * test_df["flow_m3_h"]**2
test_dataset['kilo_rpm * flow_m3_h**3'] = test_df["kilo_rpm"] * test_df["flow_m3_h"]**3

test_dataset['kilo_rpm**2'] = test_df["kilo_rpm"]**2
test_dataset['kilo_rpm**2 * flow_m3_h'] = test_df["kilo_rpm"]**2 * test_df["flow_m3_h"]
test_dataset['kilo_rpm**2 * flow_m3_h**2'] = test_df["kilo_rpm"]**2 * test_df["flow_m3_h"]**2
test_dataset['kilo_rpm**2 * flow_m3_h**3'] = test_df["kilo_rpm"]**2 * test_df["flow_m3_h"]**3

test_dataset['kilo_rpm**3'] = test_df["kilo_rpm"]**3
test_dataset['kilo_rpm**3 * flow_m3_h'] = test_df["kilo_rpm"]**3 * test_df["flow_m3_h"]
test_dataset['kilo_rpm**3 * flow_m3_h**2'] = test_df["kilo_rpm"]**3 * test_df["flow_m3_h"]**2
test_dataset['kilo_rpm**3 * flow_m3_h**3'] = test_df["kilo_rpm"]**3 * test_df["flow_m3_h"]**3

test_dataset["flow_m3_h"] = test_df["flow_m3_h"]
test_dataset["flow_m3_h**2"] = test_df["flow_m3_h"]**2
test_dataset["flow_m3_h**3"] = test_df["flow_m3_h"]**3

test_dataset["constant"] = 1

In [13]:
target = "power_w"
feature_names = [column for column in dataset.columns if column != target]
X = dataset[feature_names].to_numpy(dtype=float)
y = dataset[target].to_numpy(dtype=float)

X_test = test_dataset[feature_names].to_numpy(dtype=float)
y_test = test_dataset[target].to_numpy(dtype=float)

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

model = LassoCV(cv=5, max_iter=10000)
model.fit(X_train_scaled, y)

predictions = model.predict(X_test_scaled)

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Scale features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# Fit Lasso
lasso = LassoCV(cv=5, max_iter=100000)
lasso.fit(X_train_scaled, y)

# Predict
predictions = lasso.predict(X_test_scaled)

# Evaluate
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print(f"Chosen alpha: {lasso.alpha_}")
print(f"Test MSE:  {mse:.3f}")
print(f"Test RMSE: {rmse:.3f} W")
print(f"Test R²:   {r2:.6f}")

Chosen alpha: 0.523076799939801
Test MSE:  135.371
Test RMSE: 11.635 W
Test R²:   0.999532


In [10]:
for name, coef in zip(feature_names, lasso.coef_):
    if abs(coef) > 1e-6:
        print(f"KEPT:    {name:35} {coef:.4f}")
    else:
        print(f"REMOVED: {name}")

REMOVED: kilo_rpm
KEPT:    kilo_rpm * flow_m3_h                60.1734
REMOVED: kilo_rpm * flow_m3_h**2
REMOVED: kilo_rpm * flow_m3_h**3
KEPT:    kilo_rpm**2                         35.3698
KEPT:    kilo_rpm**2 * flow_m3_h             390.9868
REMOVED: kilo_rpm**2 * flow_m3_h**2
KEPT:    kilo_rpm**2 * flow_m3_h**3          -3.4287
KEPT:    kilo_rpm**3                         222.0098
KEPT:    kilo_rpm**3 * flow_m3_h             123.1082
REMOVED: kilo_rpm**3 * flow_m3_h**2
KEPT:    kilo_rpm**3 * flow_m3_h**3          -6.9435
REMOVED: flow_m3_h
REMOVED: flow_m3_h**2
KEPT:    flow_m3_h**3                        -304.3323
REMOVED: constant
